<a href="https://colab.research.google.com/github/LovePandey/LovePandey/blob/claude%2Fexcel-analytics-tool-1BU1L/day6_forecasting_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install anthropic scikit-learn pandas numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.2/458.2 kB 7.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

# 12 months of actuals
np.random.seed(42)
months = pd.date_range(start="2024-01-01", periods=12, freq="MS")

revenue = [500, 520, 510, 540, 560, 575, 590, 610, 600, 625, 640, 660]
cogs    = [300, 312, 308, 320, 330, 340, 348, 358, 355, 365, 374, 384]
ebitda  = [r - c for r, c in zip(revenue, cogs)]

df = pd.DataFrame({
    "month": months,
    "revenue": revenue,
    "cogs": cogs,
    "ebitda": ebitda
})

print(df.to_string(index=False))

     month  revenue  cogs  ebitda
2024-01-01      500   300     200
2024-02-01      520   312     208
2024-03-01      510   308     202
2024-04-01      540   320     220
2024-05-01      560   330     230
2024-06-01      575   340     235
2024-07-01      590   348     242
2024-08-01      610   358     252
2024-09-01      600   355     245
2024-10-01      625   365     260
2024-11-01      640   374     266
2024-12-01      660   384     276


In [ ]:
from sklearn.linear_model import LinearRegression

# Use month index as the feature (1–12 for actuals, 13–15 for forecast)
X_train = np.arange(1, 13).reshape(-1, 1)
forecast_months = np.arange(13, 16).reshape(-1, 1)
forecast_dates = pd.date_range(start="2025-01-01", periods=3, freq="MS")

forecasts = {}
for col in ["revenue", "cogs", "ebitda"]:
    model = LinearRegression()
    model.fit(X_train, df[col])
    forecasts[col] = model.predict(forecast_months).round(1)

df_forecast = pd.DataFrame({
    "month": forecast_dates,
    "revenue": forecasts["revenue"],
    "cogs": forecasts["cogs"],
    "ebitda": forecasts["ebitda"]
})

print("=== Forecast: Next 3 Months ===")
print(df_forecast.to_string(index=False))

=== Forecast: Next 3 Months ===
     month  revenue  cogs  ebitda
2025-01-01    670.9 390.0   280.9
2025-02-01    685.3 397.5   287.8
2025-03-01    699.7 405.0   294.6


In [ ]:
import anthropic

client = os.environ.get("ANTHROPIC_API_KEY")

# Summarise actuals and forecast for the prompt
actuals_summary = df[["month", "revenue", "cogs", "ebitda"]].tail(3).to_string(index=False)
forecast_summary = df_forecast.to_string(index=False)

prompt = f"""You are a senior FP&A analyst preparing a forward-looking financial commentary for the CFO.

ACTUALS (last 3 months):
{actuals_summary}

FORECAST (next 3 months, linear regression):
{forecast_summary}

Write a concise CFO-ready outlook commentary (3–4 sentences) covering:
- Revenue trajectory and growth momentum
- COGS trend and any margin implications
- EBITDA outlook and key risks to the forecast
Use professional finance language. Be specific about the numbers."""

response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=400,
    messages=[{"role": "user", "content": prompt}]
)

print("=== AI Forecast Commentary ===\n")
print(response.content[0].text)

=== AI Forecast Commentary ===

## Financial Outlook Commentary – Q1 2025

Revenue momentum remains solid, with our linear forecast projecting a trajectory from $670.9M in January to $699.7M in March, representing approximately 6% sequential growth over Q1 and sustaining the healthy month-over-month expansion (~2.5%) observed in Q4 2024. COGS is expected to rise proportionally from $390.0M to $405.0M, maintaining gross margins near 42%—consistent with recent actuals—though this assumes stable input costs and no supply chain disruptions. EBITDA is forecast to expand from $280.9M to $294.6M by quarter-end, implying margin preservation at approximately 42%; however, key downside risks include potential commodity price volatility, demand softening in a tightening macro environment, and the inherent limitation of linear projections to capture seasonal or cyclical inflection points.


In [ ]:
print("=" * 55)
print("FINANCIAL FORECAST REPORT — Q1 2025")
print("=" * 55)

print("\n📊 ACTUALS (Oct–Dec 2024)")
print(df[["month", "revenue", "cogs", "ebitda"]].tail(3).to_string(index=False))

print("\n📈 FORECAST (Jan–Mar 2025)")
print(df_forecast.to_string(index=False))

print("\n💬 AI COMMENTARY")
print("-" * 55)
print(response.content[0].text)
print("=" * 55)

FINANCIAL FORECAST REPORT — Q1 2025

📊 ACTUALS (Oct–Dec 2024)
     month  revenue  cogs  ebitda
2024-10-01      625   365     260
2024-11-01      640   374     266
2024-12-01      660   384     276

📈 FORECAST (Jan–Mar 2025)
     month  revenue  cogs  ebitda
2025-01-01    670.9 390.0   280.9
2025-02-01    685.3 397.5   287.8
2025-03-01    699.7 405.0   294.6

💬 AI COMMENTARY
-------------------------------------------------------
## Financial Outlook Commentary – Q1 2025

Revenue momentum remains solid, with our linear forecast projecting a trajectory from $670.9M in January to $699.7M in March, representing approximately 6% sequential growth over Q1 and sustaining the healthy month-over-month expansion (~2.5%) observed in Q4 2024. COGS is expected to rise proportionally from $390.0M to $405.0M, maintaining gross margins near 42%—consistent with recent actuals—though this assumes stable input costs and no supply chain disruptions. EBITDA is forecast to expand from $280.9M to $294.6M by